In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-04-10T14:12:39.896237+00:00', 'open': 101.78, 'high': 104.55, 'low': 101.09, 'close': 102.94, 'volume': 812, 'trade_count': 33, 'vwap': 102.54}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-04-10T14:12:39.896237+00:00', 'open': 94.53, 'high': 97.24, 'low': 92.39, 'close': 95.25, 'volume': 233, 'trade_count': 39, 'vwap': 94.61}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-04-10T14:12:39.896237+00:00', 'open': 96.65, 'high': 99.36, 'low': 95.56, 'close': 97.6, 'volume': 562, 'trade_count': 37, 'vwap': 97.28}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-04-10T14:12:40.917182+00:00', 'open': 103.05, 'high': 105.38, 'low': 103.11, 'close': 104.23, 'volume': 411, 'trade_count': 35, 'vwap': 104.84}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-04-10T14:12:40.917182+00:00', 'open': 100.63, 'high': 102.95, 'low': 99.43, 'close': 101.75, 'volume': 639, 'trade_count': 44, 'vwap': 102.27}
Pushed to te